# Exp5.2.2 — Single-segment dynamics visualization

Checkpoint-only diagnostic. **No training and no weight update.** The notebook traces the **FULL PADDED WINDOW** and marks the original `valid_length` in every time-axis figure. Native prediction still uses only the valid interval.

Default comparison: `s234567 + hidden_count_linear + seed11`, `normal` versus `reset567`.

In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
repo_root = next((p for p in (start, *start.parents) if (p / 'scripts').is_dir() and (p / 'notebooks').is_dir()), None)
if repo_root is None:
    raise RuntimeError(f'Could not locate writingRing repo root from cwd={start}')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from scripts import experiment_3_0_1_single_tau_objectives as base
from scripts import experiment_5_2_2_frozen_local_multitau_syn as exp522

print('repo_root:', repo_root)
print('scripts import: OK')

## Configuration and checkpoint loading

`normal_wrong_reset_correct` is preferred so the first visualization shows a segment where removing long-state continuity changes an error into a correct prediction. Set `SAMPLE_INDEX` to force an exact sample.

In [ ]:
PROFILE = 's234567'
READOUT = 'hidden_count_linear'
SEED = 11
SPLIT = 'test'
INTERVENTION = 'reset567'
SELECTION = 'normal_wrong_reset_correct'
SAMPLE_INDEX = None
DEVICE = 'cpu'
BATCH_SIZE = 128
SAVE_FIGURES = False

results_root = exp522.results_dir(repo_root)
config = exp522.Config(repo_root=repo_root, results_dir=results_root, device=DEVICE, batch_size=BATCH_SIZE, threads=1)
data = base.prepare_data(repo_root)
spec = exp522.RunSpec(PROFILE, READOUT, SEED)
cache = exp522.load_source_cache(SEED, data, config)
model, checkpoint_payload = exp522.load_model(spec, data, config)
model.eval()
device = torch.device(DEVICE)
X, y, lengths = exp522._partitions(data, cache)[SPLIT]
FULL_T = int(X.shape[1])
reset_shifts = exp522.effective_reset_shifts(PROFILE, INTERVENTION)

def predict_partition(reset_shifts_here):
    parts = []
    with torch.no_grad():
        for start_index in range(0, len(X), BATCH_SIZE):
            stop_index = min(start_index + BATCH_SIZE, len(X))
            xb = torch.as_tensor(X[start_index:stop_index], dtype=torch.float32, device=device)
            lb = torch.as_tensor(lengths[start_index:stop_index], dtype=torch.long, device=device)
            trajectory = model.forward_trajectory(xb, reset_shifts=reset_shifts_here)
            logits = model.native_logits(trajectory, lb)
            parts.append(logits.argmax(dim=1).detach().cpu().numpy())
    return np.concatenate(parts)

normal_pred = predict_partition(())
reset_pred = predict_partition(reset_shifts)

def choose_index():
    if SAMPLE_INDEX is not None:
        idx = int(SAMPLE_INDEX)
        if not 0 <= idx < len(y):
            raise IndexError(f'SAMPLE_INDEX={idx} outside 0..{len(y)-1}')
        return idx, 'explicit_sample_index'
    normal_correct = normal_pred == y
    reset_correct = reset_pred == y
    masks = {
        'normal_wrong_reset_correct': (~normal_correct) & reset_correct,
        'normal_correct_reset_wrong': normal_correct & (~reset_correct),
        'prediction_changed': normal_pred != reset_pred,
        'both_correct': normal_correct & reset_correct,
        'both_wrong': (~normal_correct) & (~reset_correct),
        'first': np.ones(len(y), dtype=bool),
    }
    priorities = [SELECTION, 'normal_wrong_reset_correct', 'prediction_changed', 'both_wrong', 'both_correct', 'first']
    seen = set()
    for name in priorities:
        if name in seen or name not in masks:
            continue
        seen.add(name)
        candidates = np.flatnonzero(masks[name])
        if candidates.size:
            return int(candidates[0]), name
    raise RuntimeError('No segment available')

sample_index, selection_reason = choose_index()
valid_length = int(lengths[sample_index])
assert 0 < valid_length <= FULL_T
sample_info = pd.DataFrame([{
    'profile': PROFILE, 'readout': READOUT, 'seed': SEED, 'split': SPLIT,
    'sample_index': sample_index, 'selection_reason': selection_reason,
    'true_label': data.labels[int(y[sample_index])],
    'normal_prediction': data.labels[int(normal_pred[sample_index])],
    f'{INTERVENTION}_prediction': data.labels[int(reset_pred[sample_index])],
    'valid_length': valid_length, 'full_padded_length': FULL_T,
    'padding_timesteps': FULL_T - valid_length, 'best_epoch': int(checkpoint_payload['best_epoch']),
}])
display(sample_info)

## Trace the FULL PADDED WINDOW

The original padded L2 trajectory is propagated through L3 for all `FULL_T` timesteps. The valid interval is not extended for classification; post-valid activity is diagnostic only.

In [ ]:
def trace_full_window(padded_l2, reset_shifts_here):
    x = torch.as_tensor(padded_l2, dtype=torch.float32, device=device).unsqueeze(0)
    if int(x.shape[1]) != FULL_T:
        raise ValueError('Expected the original full padded window')
    syn = torch.zeros(1, exp522.TEMPORAL_WIDTH, device=device, dtype=x.dtype)
    mem = torch.zeros_like(syn)
    output_mem = torch.zeros(1, model.n_classes, device=device, dtype=x.dtype)
    reset_mask = model._reset_mask(reset_shifts_here, device)
    store = {k: [] for k in [
        'hidden_input_drive', 'hidden_synaptic', 'hidden_pre_reset_membrane',
        'hidden_spikes', 'hidden_post_reset_membrane', 'pre_lif_logits',
        'output_spikes', 'output_pre_reset_membrane', 'output_post_reset_membrane'
    ]}
    with torch.no_grad():
        for t in range(FULL_T):
            if t > 0 and t % model.bin_steps == 0 and bool(reset_mask.any()):
                mask = reset_mask.unsqueeze(0)
                syn = syn.masked_fill(mask, 0.0)
                mem = mem.masked_fill(mask, 0.0)
            drive = model.input_hidden(x[:, t])
            syn = model.alpha * syn + drive
            hidden_spike, mem, hidden_pre = model.hidden_lif(syn, mem)
            class_drive = model.output_linear(hidden_spike)
            store['hidden_input_drive'].append(drive)
            store['hidden_synaptic'].append(syn)
            store['hidden_pre_reset_membrane'].append(hidden_pre)
            store['hidden_spikes'].append(hidden_spike)
            store['hidden_post_reset_membrane'].append(mem)
            store['pre_lif_logits'].append(class_drive)
            if model.output_lif is not None:
                output_spike, output_mem, output_pre = model.output_lif(class_drive, output_mem)
                store['output_spikes'].append(output_spike)
                store['output_pre_reset_membrane'].append(output_pre)
                store['output_post_reset_membrane'].append(output_mem)
    trace = {'l2_spikes': x.squeeze(0).detach().cpu().numpy()}
    for key, values in store.items():
        if values:
            trace[key] = torch.stack(values, dim=1).squeeze(0).detach().cpu().numpy()
    return trace

normal_trace = trace_full_window(X[sample_index], ())
reset_trace = trace_full_window(X[sample_index], reset_shifts)
traces = {'normal': normal_trace, INTERVENTION: reset_trace}

# Sanity check: the diagnostic forward must exactly reproduce Exp5.2.2.
xb = torch.as_tensor(X[sample_index:sample_index+1], dtype=torch.float32, device=device)
with torch.no_grad():
    for name, reset_shifts_here in [('normal', ()), (INTERVENTION, reset_shifts)]:
        expected = model.forward_trajectory(xb, reset_shifts=reset_shifts_here)
        got = traces[name]
        expected_spikes = expected['hidden_spikes'].squeeze(0).detach().cpu().numpy()
        expected_synaptic = expected['hidden_synaptic'].squeeze(0).detach().cpu().numpy()
        expected_membrane = expected['hidden_membranes'].squeeze(0).detach().cpu().numpy()
        assert np.allclose(got['hidden_spikes'], expected_spikes)
        assert np.allclose(got['hidden_synaptic'], expected_synaptic)
        assert np.allclose(got['hidden_post_reset_membrane'], expected_membrane)

def native_logits_valid(trace):
    if READOUT == 'hidden_count_linear':
        return trace['pre_lif_logits'][:valid_length].sum(axis=0)
    return trace['output_spikes'][:valid_length].sum(axis=0)

for name, trace in traces.items():
    logits = native_logits_valid(trace)
    print(name, 'VALID-only prediction =', data.labels[int(np.argmax(logits))])
print(f'valid_length={valid_length}, full padded window={FULL_T}, padding={FULL_T-valid_length}')

## Visualization helpers

Every time-axis panel marks **Valid end** with a red dashed line. The shaded region is padding; dotted vertical lines are the 250 ms reset grid.

In [ ]:
def mark_time_regions(ax):
    if valid_length < FULL_T:
        ax.axvspan(valid_length - 0.5, FULL_T - 0.5, color='0.94', zorder=-20, label='Padding')
    ax.axvline(valid_length - 0.5, color='red', linestyle='--', linewidth=2.0, label=f'Valid end = {valid_length}')
    for t in range(model.bin_steps, FULL_T, model.bin_steps):
        ax.axvline(t - 0.5, color='0.75', linestyle=':', linewidth=0.6, zorder=-10)
    ax.set_xlim(-0.5, FULL_T - 0.5)

def decorate_neuron_axis(ax):
    centers, labels = [], []
    for shift, group in exp522.profile_group_slices(PROFILE).items():
        if group.start:
            ax.axhline(group.start - 0.5, color='0.6', linewidth=0.8)
        centers.append((group.start + group.stop - 1) / 2.0)
        labels.append(f's{shift} ({group.stop-group.start})')
    ax.set_yticks(centers)
    ax.set_yticklabels(labels)
    ax.set_ylabel('L3 neuron group')

def plot_raster():
    fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True, sharey=True)
    for ax, (name, trace) in zip(axes, traces.items()):
        spikes = trace['hidden_spikes'] > 0.5
        ts, neurons = np.nonzero(spikes)
        ax.scatter(ts, neurons, s=6, c='black', marker='.')
        decorate_neuron_axis(ax); mark_time_regions(ax)
        ax.set_ylim(exp522.TEMPORAL_WIDTH - 0.5, -0.5); ax.set_title(name)
    axes[-1].set_xlabel('Timestep')
    axes[0].legend(loc='upper left', bbox_to_anchor=(1.01, 1.0))
    fig.suptitle('L3 spike raster — FULL PADDED WINDOW'); fig.tight_layout(); return fig

def plot_state_heatmap(key, title, colorbar_label):
    arrays = [trace[key].T for trace in traces.values()]
    vmax = max(max(float(np.max(np.abs(array))), exp522.THRESHOLD) for array in arrays)
    norm = SymLogNorm(linthresh=exp522.THRESHOLD, vmin=-vmax, vmax=vmax, base=10)
    fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True, sharey=True)
    image = None
    for ax, (name, trace) in zip(axes, traces.items()):
        image = ax.imshow(trace[key].T, aspect='auto', interpolation='nearest', cmap='coolwarm', norm=norm)
        decorate_neuron_axis(ax); mark_time_regions(ax); ax.set_title(name)
    axes[-1].set_xlabel('Timestep'); axes[0].legend(loc='upper left', bbox_to_anchor=(1.01, 1.0))
    fig.suptitle(title + ' — FULL PADDED WINDOW'); fig.colorbar(image, ax=list(axes), shrink=0.85, label=colorbar_label)
    fig.tight_layout(); return fig

def plot_group_firing():
    fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True, sharey=True)
    for ax, (name, trace) in zip(axes, traces.items()):
        for shift, group in exp522.profile_group_slices(PROFILE).items():
            ax.plot(trace['hidden_spikes'][:, group].mean(axis=1), label=f's{shift}')
        mark_time_regions(ax); ax.set_ylim(0, 1); ax.set_ylabel('Firing fraction'); ax.set_title(name)
    axes[-1].set_xlabel('Timestep'); axes[0].legend(ncol=6, loc='upper left', bbox_to_anchor=(1.01, 1.0))
    fig.suptitle('L3 group firing fraction — FULL PADDED WINDOW'); fig.tight_layout(); return fig

def plot_readout_activity():
    true_class = int(y[sample_index])
    if READOUT == 'hidden_count_linear':
        fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True, sharey=True)
        for ax, (name, trace) in zip(axes, traces.items()):
            cumulative = np.cumsum(trace['pre_lif_logits'], axis=0)
            for k, label in enumerate(data.labels):
                ax.plot(cumulative[:, k], linewidth=2.6 if k == true_class else 1.0, label=str(label))
            mark_time_regions(ax); ax.set_ylabel('Cumulative logit'); ax.set_title(name)
        axes[-1].set_xlabel('Timestep'); axes[0].legend(ncol=6, fontsize=8, loc='upper left', bbox_to_anchor=(1.01, 1.0))
        fig.suptitle('Cumulative class evidence — post-valid continuation is diagnostic only'); fig.tight_layout(); return fig
    fig, axes = plt.subplots(3, 2, figsize=(16, 11), sharex='col')
    for col, (name, trace) in enumerate(traces.items()):
        output_spikes = trace['output_spikes'] > 0.5
        ts, neurons = np.nonzero(output_spikes)
        axes[0, col].scatter(ts, neurons, s=10, c='black', marker='.')
        axes[0, col].set_yticks(range(len(data.labels))); axes[0, col].set_yticklabels([str(v) for v in data.labels])
        axes[0, col].set_ylabel('Output neuron'); axes[0, col].set_title(name)
        for k, label in enumerate(data.labels):
            axes[1, col].plot(trace['output_post_reset_membrane'][:, k], linewidth=2.6 if k == true_class else 1.0, label=str(label))
            axes[2, col].plot(np.cumsum(output_spikes[:, k]), linewidth=2.6 if k == true_class else 1.0, label=str(label))
        axes[1, col].axhline(exp522.THRESHOLD, color='0.5', linestyle='--', linewidth=0.8)
        axes[1, col].set_ylabel('Output post-reset U'); axes[2, col].set_ylabel('Cumulative output spikes'); axes[2, col].set_xlabel('Timestep')
        for row in range(3): mark_time_regions(axes[row, col])
    axes[1, 0].legend(ncol=3, fontsize=8, loc='upper left', bbox_to_anchor=(1.01, 1.0))
    fig.suptitle('Output neuron activity — FULL PADDED WINDOW'); fig.tight_layout(); return fig

## Figures

The red dashed line is **Valid end**. Everything to its right is padding/post-valid dynamics.

In [ ]:
figures = {
    '01_l3_spike_raster': plot_raster(),
    '02_l3_synaptic_state': plot_state_heatmap('hidden_synaptic', 'L3 signed synaptic state I', 'I'),
    '03_l3_pre_reset_membrane': plot_state_heatmap('hidden_pre_reset_membrane', 'L3 pre-reset membrane U^-', 'U^-'),
    '04_l3_post_reset_membrane': plot_state_heatmap('hidden_post_reset_membrane', 'L3 post-reset membrane U', 'U'),
    '05_l3_group_firing': plot_group_firing(),
    '06_readout_activity': plot_readout_activity(),
}
if SAVE_FIGURES:
    out_dir = results_root / 'single_segment_dynamics_notebook' / spec.key / f'{SPLIT}_sample{sample_index:04d}__normal_vs_{INTERVENTION}'
    out_dir.mkdir(parents=True, exist_ok=True)
    for name, fig in figures.items():
        fig.savefig(out_dir / f'{name}.png', dpi=180, bbox_inches='tight')
    print('saved figures to', out_dir)
plt.show()

## Compact valid-vs-padding activity summary

This table is secondary to the figures. `padding_firing_rate_hz` and `padding_spike_fraction_of_full_window` quantify residual activity after the original valid interval.

In [ ]:
def run_lengths(binary):
    runs, current = [], 0
    for value in np.asarray(binary, dtype=bool):
        if value:
            current += 1
        elif current:
            runs.append(current); current = 0
    if current:
        runs.append(current)
    return runs

rows = []
for condition, trace in traces.items():
    for shift, group in exp522.profile_group_slices(PROFILE).items():
        spikes = trace['hidden_spikes'][:, group] > 0.5
        post_mem = trace['hidden_post_reset_membrane'][:, group]
        valid_spikes = spikes[:valid_length]
        padding_spikes = spikes[valid_length:]
        full_runs = [run for neuron in range(spikes.shape[1]) for run in run_lengths(spikes[:, neuron])]
        valid_runs = [run for neuron in range(spikes.shape[1]) for run in run_lengths(valid_spikes[:, neuron])]
        valid_positions = np.nonzero(valid_spikes)
        backlog = np.nan if len(valid_positions[0]) == 0 else float(np.mean(post_mem[:valid_length][valid_positions] > exp522.THRESHOLD))
        total_spikes = int(spikes.sum())
        padding_spike_count = int(padding_spikes.sum())
        rows.append({
            'condition': condition, 'shift': shift,
            'valid_firing_rate_hz': float(valid_spikes.mean() * data.fs),
            'padding_firing_rate_hz': float(padding_spikes.mean() * data.fs) if padding_spikes.size else np.nan,
            'padding_spike_fraction_of_full_window': padding_spike_count / max(total_spikes, 1),
            'max_run_length_valid': max(valid_runs, default=0),
            'max_run_length_full': max(full_runs, default=0),
            'post_reset_U_above_threshold_given_valid_spike': backlog,
            'endpoint_mean_abs_I': float(np.mean(np.abs(trace['hidden_synaptic'][valid_length-1, group]))),
            'endpoint_mean_abs_U': float(np.mean(np.abs(trace['hidden_post_reset_membrane'][valid_length-1, group]))),
        })
activity_summary = pd.DataFrame(rows)
display(activity_summary)

## Reading the figures

- Long horizontal black bands in `s6/s7` indicate sustained firing rather than sparse events.
- Bands continuing beyond **Valid end** show residual post-valid activity.
- Large positive `hidden_pre_reset_membrane` together with post-reset membrane remaining above threshold after a spike supports a binary-cap backlog interpretation.
- Compare `normal` with `reset567` to see whether resetting shifts 5/6/7 removes that sustained state.
- `Cumulative class evidence` is plotted over the full padded window only for diagnosis; native prediction above is valid-only.